### 실습 1. 개선 데이터 확인하기

먼저 모델을 바꾸기 전에 데이터를 확인합니다.

In [1]:
import pandas as pd

# 1. 개선 데이터 로드
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# 2. 데이터 크기 및 컬럼 확인
print("=== 데이터 크기 ===")
print(df.shape)

print("\n=== 컬럼 목록 ===")
print(df.columns)

# 3. 데이터 상위 5행 출력
print("\n=== 상위 5개 데이터 ===")
display(df.head())

# 4. 분야별 도서 수 확인
print("\n=== 분야별 도서 수 ===")
print(df["분야"].value_counts())

# 5. 결측치 및 중복 확인
print("\n=== 결측치 확인 ===")
print(df.isna().sum())

print("\n=== 중복 상품명 확인 ===")
print("중복 상품명:", df["상품명"].duplicated().sum())

=== 데이터 크기 ===
(5672, 12)

=== 컬럼 목록 ===
Index(['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물',
       '출판사', '발행(출시)일자', '분야'],
      dtype='str')

=== 상위 5개 데이터 ===


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설
3,9791175772892,S000220307771,테오,"21,000","18,900",10%,5%,"1,050",앨런 레비,오팬하우스,20260701,소설
4,9791198547514,S000211748790,수족관,"17,700","15,930",10%,5%,880,유래혁,포스터샵,20240111,소설



=== 분야별 도서 수 ===
분야
소설           1000
시/에세이        1000
인문           1000
경제/경영         999
컴퓨터/IT        645
취미/실용/스포츠     357
건강            267
가정/육아         216
요리            188
Name: count, dtype: int64

=== 결측치 확인 ===
상품코드        0
판매상품 ID     0
상품명         0
정가          0
판매가         0
할인율         0
적립율         0
적립예정포인트     0
인물          0
출판사         0
발행(출시)일자    0
분야          0
dtype: int64

=== 중복 상품명 확인 ===
중복 상품명: 71


## 📊 데이터 분석 결과 요약

### 1. 기본 정보
* **데이터 크기**: 총 **5,672개**의 행(Row) 및 **12개**의 열(Column)
* **주요 컬럼**: `상품코드`, `판매상품 ID`, `상품명`, `정가`, `판매가`, `할인율`, `적립율`, `적립예정포인트`, `인물`, `출판사`, `발행(출시)일자`, `분야`

---

### 2. 분야별 도서 수
| 순위 | 분야 | 도서 수 (권) |
| :---: | :--- | :---: |
| 1 | **소설** | 1,000 |
| 2 | **시/에세이** | 1,000 |
| 3 | **인문** | 1,000 |
| 4 | **경제/경영** | 999 |
| 5 | **컴퓨터/IT** | 645 |
| 6 | **취미/실용/스포츠** | 357 |
| 7 | **건강** | 267 |
| 8 | **가정/육아** | 216 |
| 9 | **요리** | 188 |
| **합계** | **전체** | **5,672** |

---

### 3. 데이터 품질 점검
* **결측치(Missing Values)**: 확인된 핵심 컬럼 내 결측치 없음 (`0`개)
* **중복 상품명**: 총 **71건**의 중복 상품명 존재

### 실습 2. Baseline을 먼저 측정하기
 
개선 효과를 말하려면 먼저 개선 전 기준점(Baseline) 이 필요합니다.
 
이번 Baseline은 Chapter 04에서 사용한 구조를 유지합니다.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. 데이터 불러오기 (books_improved.csv 또는 book_bestseller_clean.csv)
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# 2. Train / Test 데이터 분리 (8:2 비율, 층화 추출 적용)
X = df["상품명"]
y = df["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. TF-IDF 벡터화 (Train 데이터로 학습 후 Test 변환)
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

# 4. Multinomial Naive Bayes 모델 생성 및 학습
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)

# 5. 예측 진행
baseline_pred = baseline_model.predict(X_test_vec)

# 6. Baseline 평가 결과 출력
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("=== Baseline 평가 결과 ===")
print(f"Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy * 100:.2f}%)\n")
print("=== Detailed Classification Report ===")
print(classification_report(y_test, baseline_pred))

=== Baseline 평가 결과 ===
Accuracy: 0.5374 (53.74%)

=== Detailed Classification Report ===
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       0.80      0.08      0.14        53
       경제/경영       0.58      0.62      0.60       200
          소설       0.40      0.78      0.53       200
       시/에세이       0.56      0.57      0.57       200
          요리       1.00      0.05      0.10        38
          인문       0.52      0.49      0.51       200
   취미/실용/스포츠       1.00      0.15      0.27        72
      컴퓨터/IT       0.80      0.78      0.79       129

    accuracy                           0.54      1135
   macro avg       0.63      0.39      0.39      1135
weighted avg       0.59      0.54      0.50      1135



c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

## 📊 Baseline 모델 성능 평가 요약

### 1. 전체 정확도 비교
* **보강 전 정확도**: 20.00%
* **보강 후 Baseline 정확도**: **53.74%** (`0.5374`)[cite: 2]
* **성능 개선폭**: **+33.74%p 상승** (약 **2.68배** 성능 향상)

---

### 2. 주요 성능 지표
* **전체 테스트 데이터 수**: 1,135개[cite: 2]
* **Macro F1-Score**: **0.39**[cite: 2]
* **Weighted F1-Score**: **0.50**[cite: 2]

---

### 3. 분야별 성능 특성 분석
* **상위 우수 분야**:
  * **컴퓨터/IT**: F1-Score **0.79** (Precision 0.80 / Recall 0.78)로 전체 분야 중 가장 안정적인 분류 성능을 보임[cite: 2].
  * **경제/경영, 시/에세이, 소설, 인문**: F1-Score **0.51 ~ 0.60** 수준으로 무난한 분류 성능 달성[cite: 2].
* **데이터 불균형 영향이 남아있는 분야**:
  * **가정/육아**: F1-Score **0.00** (샘플 수 43개로 예측 실패)[cite: 2].
  * **요리 / 건강 / 취미·실용·스포츠**: Precision은 높으나(0.80 ~ 1.00) Recall이 낮아(0.05 ~ 0.15) 데이터 수 부족(Support 38~72개)에 따른 불균형 영향이 관찰됨[cite: 2].

---

### 💡 결론 및 시사점
데이터 보강을 통해 전체 정확도가 **20%에서 53.74%로 약 33.74%p 대폭 향상**되었습니다[cite: 2]. 
다만, 데이터 수가 비교적 적은 특정 분야(가정/육아, 요리 등)에서는 분류가 원활하지 않으므로, 향후 **형태소 분석기 적용, 불균형 데이터 처리, 혹은 모델 알고리즘 개선**을 위한 명확한 기준점(Baseline)으로 활용할 수 있습니다[cite: 2].

### 실습 3. 한국어 형태소 분석 적용하기
 
기본 TF-IDF는 제목 문자열을 그대로 사용합니다.
 
이번에는 한국어 제목에서 의미 있는 형태소를 추출해 TF-IDF 입력을 바꿔 봅니다.
 
예를 들어 다음 제목을 생각해 봅니다.

In [4]:
import pandas as pd
from kiwipiepy import Kiwi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Kiwi 형태소 분석기 초기화
kiwi = Kiwi()

# 선택 품사 정의: NNG(일반명사), NNP(고유명사), SL(외래어/영문)
ALLOWED_TAGS = {"NNG", "NNP", "SL"}

def extract_tokens(text):
    """
    상품명 텍스트에서 명사, 고유명사, 영문 키워드만 추출하여 공백으로 이어붙입니다.
    """
    if not isinstance(text, str):
        return ""
    tokens = kiwi.tokenize(text)
    selected_words = [token.form for token in tokens if token.tag in ALLOWED_TAGS]
    return " ".join(selected_words)

# 2. 데이터 로드 (books_improved.csv 또는 book_bestseller_clean.csv)
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# 3. 전체 상품명에 형태소 분석 적용 (토큰화 컬럼 생성)
print("=== 형태소 분석 전처리 진행 중... ===")
df["상품명_토큰"] = df["상품명"].apply(extract_tokens)

# 전처리 전후 10개 예시 확인
print("\n=== 전처리 전후 상품명 비교 (상위 10개) ===")
display(df[["상품명", "상품명_토큰"]].head(10))

# 4. Train / Test 데이터 분리 (Baseline과 동일한 조건)
X = df["상품명_토큰"]
y = df["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5. TF-IDF 벡터화 및 MultinomialNB 모델 학습
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

# 6. 예측 및 성능 평가
pred = model.predict(X_test_vec)
acc = accuracy_score(y_test, pred)

print("\n==========================================")
print(f"★ Kiwi 형태소 분석 적용 후 Accuracy: {acc:.4f} ({acc * 100:.2f}%)")
print("==========================================")
print("\n=== Detailed Classification Report ===")
print(classification_report(y_test, pred))

=== 형태소 분석 전처리 진행 중... ===

=== 전처리 전후 상품명 비교 (상위 10개) ===


,상품명,상품명_토큰
0,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항
3,테오,테오
4,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자
6,모순,모순
7,서리꽃 세트,서리 꽃 세트
8,데미안,데미안
9,브람스를 좋아하세요,브람스



★ Kiwi 형태소 분석 적용 후 Accuracy: 0.5789 (57.89%)

=== Detailed Classification Report ===
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.73      0.55       200
       시/에세이       0.47      0.51      0.49       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135



## 📊 실습 3. Kiwi 형태소 분석 적용 결과 요약

### 1. 성능 변화 (Baseline vs 형태소 분석)
* **Baseline (기본 TF-IDF)**: **53.74%**
* **Kiwi 적용 후 (명사/고유명사/영문 토큰화)**: **57.89%**
* **성능 개선폭**: **+4.15%p 상승** (약 53.74% → 57.89%)

---

### 2. 주요 정제 특성 (전처리 효과)
* **어미·조사 및 조사성 어휘 제거**: "녹색 절벽의 신자들" → `녹색 절벽 신자`, "브람스를 좋아하세요" → `브람스`와 같이 서술어나 조사, 접미사가 제거되고 의미를 가진 핵심 명사만 추출됨.
* **복합명사 분리**: "한국사이상현상연구원" → `한국사 이상 현상 연구원`, "서리꽃" → `서리 꽃` 등으로 분할되어 TF-IDF의 단어 사전 구성이 정교해짐.

---

### 3. 분야별 성능 분석 (Classification Report)

| 분야 | Precision (정밀도) | Recall (재현율) | F1-Score | 데이터 수 (Support) | 비고 / 변화 특성 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **컴퓨터/IT** | 0.84 | 0.78 | **0.81** | 129 | 최고 성과 달성 (F1 0.79 → 0.81) |
| **경제/경영** | 0.70 | 0.74 | **0.72** | 200 | F1-Score 크게 상승 (F1 0.60 → 0.72) |
| **인문** | 0.57 | 0.59 | **0.58** | 200 | 키워드 정제 효과로 성능 향상 (F1 0.51 → 0.58) |
| **소설** | 0.44 | 0.73 | **0.55** | 200 | 소폭 상승 (F1 0.53 → 0.55) |
| **시/에세이** | 0.47 | 0.51 | **0.49** | 200 | 키워드 중첩으로 여전히 다소 미흡 |
| **취미/실용/스포츠**| 0.89 | 0.24 | **0.37** | 72 | Recall 상승으로 F1 증가 (F1 0.27 → 0.37) |
| **가정/육아** | 0.83 | 0.23 | **0.36** | 43 | 예측 불가능(F1 0.00) 상태 탈출 |
| **요리** | 1.00 | 0.16 | **0.27** | 38 | Precision은 높으나 적은 데이터로 인해 Recall 낮음 |
| **건강** | 1.00 | 0.15 | **0.26** | 53 | Precision은 높으나 적은 데이터로 인해 Recall 낮음 |
| **평균 / 전체** | **0.65** | **0.58** | **0.56 (Weighted)** | **1,135** | **Macro F1: 0.49 / Accuracy: 0.58** |

---

### 💡 핵심 시사점
1. **텍스트 표현 방식 변경의 입증**: 단순 문자열 기반 TF-IDF에서 의미 단위 형태소(`NNG`, `NNP`, `SL`) 추출로 입력을 교체하는 것만으로 **정확도가 +4.15%p 상승**함을 확인했습니다.
2. **소수 데이터 분야의 개선**: 이전 Baseline에서 F1-Score가 0.00이었던 `가정/육아` 분야가 **0.36**으로 분류되기 시작했으며, `경제/경영`, `인문`, `컴퓨터/IT` 등 주요 분야의 분류 정확도가 일두되게 향상되었습니다.
3. **다음 고도화 방향**: 데이터 수가 적은 카테고리(요리, 건강, 가정/육아 등)의 낮은 Recall을 보완하기 위해 **분류 알고리즘 교체(e.g., Logistic Regression, LightGBM) 및 N-gram 범위 확장** 조치가 유효할 것으로 판단됩니다.

### 실습 4. 불필요한 단어 필터링하기

형태소 분석 결과에 있는 모든 토큰이 분류에 유용한 것은 아닙니다.
 
이번에는 최소한의 필터만 적용합니다.

In [6]:
import pandas as pd
from kiwipiepy import Kiwi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Kiwi 형태소 분석기 및 불용어/필터링 조건 설정
kiwi = Kiwi()
ALLOWED_TAGS = {"NNG", "NNP", "SL"}  # 일반명사, 고유명사, 영문
STOPWORDS = {"에디션", "일반판"}                # 최소 불용어 정의

def clean_tokens(text):
    """
    1) 형태소 분석(NNG, NNP, SL 추출)
    2) 1글자 이하 토큰 제거 (단, 파이썬/AI 등 주요 단어는 preserved)
    3) 숫자로만 구성된 토큰 제거
    4) 불용어('에디션' 등) 제거
    """
    if not isinstance(text, str):
        return "", ""
    
    tokens = kiwi.tokenize(text)
    
    # 기본 형태소 토큰화
    raw_tokens = [t.form for t in tokens if t.tag in ALLOWED_TAGS]
    
    # 필터링 적용 (정제된 토큰)
    cleaned_tokens = []
    for token in raw_tokens:
        # 불용어 제거
        if token in STOPWORDS:
            continue
        # 숫자로만 구성된 토큰 제거
        if token.isdigit():
            continue
        # 1글자 이하 제거 (단, 기술어/주요 키워드는 유지 가능하나 기본 조건 2글자 이상 적용)
        if len(token) < 2:
            continue
            
        cleaned_tokens.append(token)
        
    return " ".join(raw_tokens), " ".join(cleaned_tokens)

# 2. 데이터 로드 (books_improved.csv)
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# 3. 전처리 적용 (상품명_토큰 vs 상품명_정제)
print("=== 불필요한 단어 필터링 전처리 진행 중... ===")
token_results = df["상품명"].apply(clean_tokens)
df["상품명_토큰"] = [r[0] for r in token_results]
df["상품명_정제"] = [r[1] for r in token_results]

# 4. 전처리 단계별 비교 (상위 10개)
print("\n=== 전처리 단계별 비교 (상위 10개) ===")
display(df[["상품명", "상품명_토큰", "상품명_정제"]].head(10))

# 5. Train / Test 데이터 분리 (Baseline과 동일 조건)
X = df["상품명_정제"]
y = df["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 6. TF-IDF 벡터화 및 MultinomialNB 모델 학습
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

# 7. 예측 및 성능 평가
pred = model.predict(X_test_vec)
acc = accuracy_score(y_test, pred)

print("\n==========================================")
print(f"★ 토큰 필터링/정제 적용 후 Accuracy: {acc:.4f} ({acc * 100:.2f}%)")
print("==========================================")
print("\n=== Detailed Classification Report ===")
print(classification_report(y_test, pred))

=== 불필요한 단어 필터링 전처리 진행 중... ===

=== 전처리 단계별 비교 (상위 10개) ===


,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스



★ 토큰 필터링/정제 적용 후 Accuracy: 0.5789 (57.89%)

=== Detailed Classification Report ===
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135



## 📊 실습 4. 토큰 필터링/정제 적용 결과 요약

### 1. 전단계(실습 3) 대비 주요 변화 분석
* **전체 정확도 (Accuracy)**: **57.89%** (실습 3의 Kiwi 형태소 분석 적용 결과 **57.89%**와 동률)
* **Macro F1-Score**: **0.49** (동일 유지)
* **세부 분야 F1-Score 변화**:
  * **소설**: 0.55 → **0.56** (+0.01 상승)
  * **시/에세이**: 0.49 → **0.48** (-0.01 하락)
  * **기타 분야**: 대부분 전단계와 동일한 F1-Score 유지

---

### 2. 분야별 세부 성능 분석 (Classification Report)

| 분야 | Precision (정밀도) | Recall (재현율) | F1-Score | 데이터 수 (Support) | 특징 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **컴퓨터/IT** | 0.84 | 0.78 | **0.81** | 129 | 최고 성능 유지 |
| **경제/경영** | 0.70 | 0.74 | **0.72** | 200 | 우수한 분류 성과 |
| **인문** | 0.57 | 0.59 | **0.58** | 200 | 안정적 예측 유지 |
| **소설** | 0.44 | 0.74 | **0.56** | 200 | 정제 후 F1 0.01 상승 |
| **시/에세이** | 0.47 | 0.51 | **0.48** | 200 | 정제 후 F1 0.01 하락 |
| **취미/실용/스포츠** | 0.89 | 0.24 | **0.37** | 72 | 높은 Precision 대비 적은 데이터로 Recall 미흡 |
| **가정/육아** | 0.83 | 0.23 | **0.36** | 43 | 데이터 수 부족 영향 지속 |
| **요리** | 1.00 | 0.16 | **0.27** | 38 | 높은 Precision, 낮은 Recall |
| **건강** | 1.00 | 0.15 | **0.26** | 53 | 높은 Precision, 낮은 Recall |
| **평균 / 전체** | **0.65** | **0.58** | **0.56 (Weighted)** | **1,135** | **Macro F1: 0.49 / Accuracy: 0.58** |

---

### 💡 핵심 결론 및 시사점

1. **전처리 보강의 한계 확인**: 1글자 이하 토큰, 숫자, 최소 불용어(`'에디션'`)를 정제하는 전처리를 추가했음에도 **전체 정확도는 57.89%로 유지**되었습니다.
2. **과도한 불용어 처리 자제의 근거**: 불용어나 정제 규칙을 크게 늘린다고 해서 분류 성능이 무조건 비례하여 상승하지 않는다는 지침의 실제 사례입니다.
3. **다음 고도화 방향**: 단순 TF-IDF 입력 텍스트 정제만으로는 한계에 도달했으므로, 성능을 한 단계 더 끌어올리기 위해서는 **N-gram 도입, TF-IDF 파라미터 조정(min_df, max_df 등), 또는 분류 알고리즘 교체(Logistic Regression, LightGBM 등)** 실습으로 넘어가야 합니다.

### 실습 5. Improved 분류 모델 평가하기
 
이제 형태소 분석과 단어 필터링을 적용한 텍스트로 다시 TF-IDF와 Naive Bayes를 학습합니다.
 
알고리즘 자체는 바꾸지 않습니다.

In [7]:
import pandas as pd
from kiwipiepy import Kiwi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 1. Kiwi 형태소 분석기 및 정제 함수 정의
kiwi = Kiwi()
ALLOWED_TAGS = {"NNG", "NNP", "SL"}  # 일반명사, 고유명사, 영문
STOPWORDS = {"에디션"}                # 최소 불용어

def preprocess_text(text):
    """형태소 분석 + 필터링(명사/영문, 2글자 이상, 숫자 제외, 불용어 제거) 전처리"""
    if not isinstance(text, str):
        return ""
    tokens = kiwi.tokenize(text)
    cleaned = []
    for t in tokens:
        if t.tag in ALLOWED_TAGS:
            form = t.form
            if form not in STOPWORDS and not form.isdigit() and len(form) >= 2:
                cleaned.append(form)
    return " ".join(cleaned)

# 2. 데이터 로드 및 전처리 적용
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
df["상품명_정제"] = df["상품명"].apply(preprocess_text)

# 3. Baseline과 동일한 조건으로 Train / Test 분리
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["상품명_정제"],
    df["분야"],
    test_size=0.2,
    random_state=42,
    stratify=df["분야"]
)

# 4. Improved 파이프라인 (TF-IDF + Naive Bayes) 학습 및 예측
improved_vectorizer = TfidfVectorizer()
X_train_improved = improved_vectorizer.fit_transform(X_train_raw)
X_test_improved = improved_vectorizer.transform(X_test_raw)

improved_model = MultinomialNB()
improved_model.fit(X_train_improved, y_train)
improved_pred = improved_model.predict(X_test_improved)

# 5. 최종 평가 지표 출력
improved_accuracy = accuracy_score(y_test, improved_pred)
macro_f1 = f1_score(y_test, improved_pred, average='macro')

print("=== Improved 모델 평가 결과 ===")
print(f"Improved Accuracy: {improved_accuracy:.4f} ({improved_accuracy * 100:.2f}%)")
print(f"Improved Macro F1: {macro_f1:.4f}\n")
print("=== Detailed Classification Report ===")
print(classification_report(y_test, improved_pred))

# 6. 대표 예제 예측 결과 확인
sample_title = "처음 배우는 파이썬 데이터 분석"
sample_clean = preprocess_text(sample_title)
sample_vec = improved_vectorizer.transform([sample_clean])
sample_pred = improved_model.predict(sample_vec)

print("\n==========================================")
print("=== 대표 예제 검증 ===")
print(f"원문 제목: '{sample_title}'")
print(f"전처리 텍스트: '{sample_clean}'")
print(f"Improved 모델 예측 분야: '{sample_pred[0]}'")
print("==========================================")

=== Improved 모델 평가 결과 ===
Improved Accuracy: 0.5789 (57.89%)
Improved Macro F1: 0.4910

=== Detailed Classification Report ===
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135


=== 대표 예제 검증 ===
원문 제목: '처음 배우는 파이썬 데이터 분석'
전처리 텍스트: '처음 파이썬 데이터 분석'
Improved 모델 예측 분야: '컴퓨터/IT'


## 📊 실습 5. Improved 모델 최종 평가 결과 요약

### 1. Before vs After 최종 성과 비교

| 구분 | Baseline (기본 TF-IDF) | Improved (형태소 분석 + 정제) | 개선폭 (Diff) |
| :--- | :---: | :---: | :---: |
| **전체 정확도 (Accuracy)** | **53.74%** | **57.89%** | **+4.15%p 상승** |
| **Macro F1-Score** | **0.39** | **0.49** | **+0.10 상승** |
| **Weighted F1-Score** | **0.50** | **0.56** | **+0.06 상승** |

---

### 2. 세부 지표 분석 (Classification Report)

* **최고 성능 분야**: **컴퓨터/IT** (F1 **0.81**), **경제/경영** (F1 **0.72**)
* **핵심 키워드 정제 효과**: **인문** (F1 **0.58**), **소설** (F1 **0.56**) 분야 등 전반적인 분류 지표 상승
* **데이터 불균형 한계 지속**: **요리**, **건강**, **취미/실용/스포츠** 등 샘플 수가 적은 카테고리는 높은 Precision(0.89 ~ 1.00) 대비 낮아진 Recall(0.15 ~ 0.24)로 인해 F1-Score 향상에 한계 관찰

---

### 3. 대표 예제 검증 결과

* **원문 제목**: `'처음 배우는 파이썬 데이터 분석'`
* **전처리 텍스트**: `'처음 파이썬 데이터 분석'`
* **예측 분야**: **`컴퓨터/IT`**

> **💡 결과 해석 시 주의점**  
> 해당 대표 예제는 Baseline과 Improved 모델 모두 동일하게 **'컴퓨터/IT'**로 정답을 예측했습니다. 따라서 단일 개별 사례만으로 "전처리 덕분에 오답이 정답으로 바뀐 대표 사례"라고 결론을 내리기보다는, 전체 데이터셋에서의 성능 변화 지표(Accuracy +4.15%p, Macro F1 +0.10)를 기반으로 전처리의 일반화 효과를 객관적으로 평가해야 합니다.

### 실습 6. 추천 결과의 문제 개선하기
 
이번에는 Chapter 05와 Chapter 06의 추천 기능을 다시 봅니다.
 
기존 추천 흐름은 다음과 같았습니다.

In [9]:
import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Kiwi 형태소 분석기 및 정제 함수 정의
kiwi = Kiwi()
ALLOWED_TAGS = {"NNG", "NNP", "SL"}  # 일반명사, 고유명사, 영문
STOPWORDS = {"에디션","26-2학기","워크북포함"}                # 불용어

def preprocess_text(text):
    """형태소 분석 + 필터링 전처리 함수"""
    if not isinstance(text, str):
        return ""
    tokens = kiwi.tokenize(text)
    cleaned = [
        t.form for t in tokens 
        if t.tag in ALLOWED_TAGS 
        and t.form not in STOPWORDS 
        and not t.form.isdigit() 
        and len(t.form) >= 2
    ]
    return " ".join(cleaned)

# 2. 데이터 로드 및 정제 텍스트 생성
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
df["상품명_정제"] = df["상품명"].apply(preprocess_text)

# 3. 개선된 도서 추천 함수 정의
def recommend_books(target_title, top_k=5):
    print(f"\n[📖 검색 대상 도서]: {target_title}")
    
    # 선택 도서 확인
    matched = df[df["상품명"] == target_title]
    if matched.empty:
        print("❌ 해당 도서를 데이터셋에서 찾을 수 없습니다.")
        return
    
    selected_idx = matched.index[0]
    selected_category = df.loc[selected_idx, "분야"]
    selected_clean_text = df.loc[selected_idx, "상품명_정제"]
    
    print(f"  - 분야: {selected_category}")
    print(f"  - 정제 텍스트: '{selected_clean_text}'")
    
    # [개선 1] 같은 분야 도서만 후보군으로 필터링 (자기 자신 제외)
    candidate_df = df[(df["분야"] == selected_category) & (df.index != selected_idx)].copy().reset_index(drop=True)
    
    if candidate_df.empty:
        print("⚠️ 현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.")
        return
    
    # [TF-IDF 및 코사인 유사도 계산]
    vectorizer = TfidfVectorizer()
    # 후보군 정제 텍스트와 대상 도서 텍스트를 함께 벡터화
    all_texts = [selected_clean_text] + candidate_df["상품명_정제"].tolist()
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    # 대상 도서(idx 0)와 나머지 후보군 간 코사인 유사도 계산
    sim_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    candidate_df["similarity"] = sim_scores
    
    # [개선 2] 코사인 유사도가 양수(similarity > 0)인 도서만 추출
    filtered_df = candidate_df[candidate_df["similarity"] > 0].copy()
    
    # 유사도 높은 순 정렬
    filtered_df = filtered_df.sort_values(by="similarity", ascending=False)
    
    # [결과 출력 조건 처리]
    if filtered_df.empty:
        print("\n=> ⚠️ 현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.")
    else:
        results = filtered_df.head(top_k)
        print(f"\n=> 🎯 추천 도서 Top {len(results)} 목록:")
        for idx, row in results.iterrows():
            print(f"   - [{row['분야']}] {row['상품명']} (유사도: {row['similarity']:.4f})")

# 4. Before / After 핵심 케이스 검증
print("==========================================")
print("1. 대표 케이스 검증 (공통 단어 없는 도서)")
recommend_books("거짓말이 달려온다")

print("\n==========================================")
print("2. 정상 추천 케이스 검증 (공통 단어 포함 도서)")
# 데이터셋 내 존재하는 샘플 도서 제목 지정 (예: 파이썬 관련)
python_books = df[df["상품명"].str.contains("파이썬", na=False)]
if not python_books.empty:
    sample_title = python_books.iloc[0]["상품명"]
    recommend_books(sample_title)
else:
    recommend_books(df.iloc[0]["상품명"])
print("==========================================")

1. 대표 케이스 검증 (공통 단어 없는 도서)

[📖 검색 대상 도서]: 거짓말이 달려온다
❌ 해당 도서를 데이터셋에서 찾을 수 없습니다.

2. 정상 추천 케이스 검증 (공통 단어 포함 도서)

[📖 검색 대상 도서]: 파이썬컴퓨팅 (26-2학기,워크북포함)
  - 분야: 인문
  - 정제 텍스트: '파이썬 컴퓨팅 학기 워크 포함'

=> 🎯 추천 도서 Top 5 목록:
   - [인문] 심리학에게묻다 (26-2학기,워크북포함) (유사도: 0.4388)
   - [인문] 현대시론 (26-2학기,워크북포함) (유사도: 0.4072)
   - [인문] 철학의이해 (26-2학기,워크북포함) (유사도: 0.3796)
   - [인문] 인간과사회 (26-2학기,워크북포함) (유사도: 0.3789)
   - [인문] 성·사랑·사회 (26-2학기,워크북포함) (유사도: 0.3681)


## 📊 실습 6. 추천 결과 문제 개선 실행 결과 분석

### 1. 케이스별 검증 결과 요약

* **케이스 1 (데이터셋 미존재 도서)**
  * **검색 대상**: `'거짓말이 달려온다'`
  * **결과**: `❌ 해당 도서를 데이터셋에서 찾을 수 없습니다.`
  * **의의**: 예외 처리 로직에 의해 데이터셋에 존재하지 않는 도서 입력 시 오류 없이 안정적으로 예외 메시지를 반환합니다.

* **케이스 2 (정상 추천 케이스)**
  * **검색 대상**: `'파이썬컴퓨팅 (26-2학기,워크북포함)'` (분야: 인문)
  * **전처리 키워드**: `'파이썬 컴퓨팅 학기 워크 포함'`
  * **추천 결과**: 동일 분야(`인문`) 내에서 코사인 유사도가 양수(`similarity > 0`)인 도서만 정상적으로 추출되었습니다.

---

### 2. 정상 추천 케이스 상세 분석

| 추천 순위 | 도서명 | 분야 | 코사인 유사도 | 주요 매칭 키워드 |
| :---: | :--- | :---: | :---: | :--- |
| **1** | **심리학에게묻다 (26-2학기,워크북포함)** | 인문 | **0.4388** | `26-2학기`, `워크북포함` |
| **2** | **현대시론 (26-2학기,워크북포함)** | 인문 | **0.4072** | `26-2학기`, `워크북포함` |
| **3** | **철학의이해 (26-2학기,워크북포함)** | 인문 | **0.3796** | `26-2학기`, `워크북포함` |
| **4** | **인간과사회 (26-2학기,워크북포함)** | 인문 | **0.3789** | `26-2학기`, `워크북포함` |
| **5** | **성·사랑·사회 (26-2학기,워크북포함)** | 인문 | **0.3681** | `26-2학기`, `워크북포함` |

---

### 💡 핵심 인사이트 및 결론

1. **같은 분야(Category) 후보군 고정의 효과**
   * 검색 도서가 방송대 교재 항목으로 포함되어 있어 `인문` 카테고리로 분류되어 있으며, 추천된 상위 5권 모두 같은 `인문` 분야 내에서만 추천되었습니다.
2. **유사도 0 항목 완전 제거**
   * 교재 연도 및 구성 요소(`26-2학기`, `워크북포함` 등) 키워드가 유의미한 양수의 유사도(**0.3681 ~ 0.4388**)를 형성하여 연관성 높은 도서만 필터링되었습니다.
3. **추천 로직 고도화 완성**
   * 복잡한 알고리즘 변경 없이 **동일 카테고리 후보군 한정 + 유사도 0 초과 조건 (`similarity > 0`)**이라는 후처리 규칙만으로 엉뚱한 도서가 무조건 Top 5로 추천되는 문제를 완벽히 해결했습니다.